# 예제 02. 실패 유형 분류와 JSON 파서 만들기

전산실험 2 · AI Agent 만들기 · **3주차** · 예상 소요 35분

## 목표
- 실패 응답을 **수집하고 유형별로 분류**한다
- 어디까지 파서로 흡수하고 어디부터 재시도할지 판단한다
- `extract_json()` 을 직접 구현한다
- **관대한 파서의 위험**을 이해한다

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)"""
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 --------------------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용."""
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다."""
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과."""
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 ------------------------------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다."""
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다."""
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다."""
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile structured.py
"""전산실험2 3주차 : 구조화 출력 — 파싱 · 검증 · 재시도.

소형 모델은 "JSON으로만 답하라"고 지시해도 자주 어긴다.
코드펜스로 감싸거나, 인사를 덧붙이거나, 따옴표를 빠뜨린다.

이 모듈은 세 겹의 방어를 제공한다.

    1) 프롬프트로 형식을 강제하고        (build_json_prompt)
    2) 지저분한 응답에서 JSON을 긁어내고  (extract_json)
    3) 스키마로 검증한 뒤, 실패하면       (validate)
       오류를 되먹여 다시 요구한다        (ask_json)

사용 예:
    from kmu_llm import LLM
    from structured import ask_json

    schema = {
        "required": ["test", "reason"],
        "types": {"test": str, "reason": str},
        "allowed": {"test": ["t-test", "anova", "chi-square"]},
    }
    result = ask_json("두 그룹의 평균 비교에 쓸 검정은?", schema=schema)
    print(result["test"])
"""

from __future__ import annotations

import json
import re
from typing import Any

__all__ = [
    "JSONParseError",
    "SchemaError",
    "extract_json",
    "parse_json",
    "validate",
    "build_json_prompt",
    "ask_json",
]


class JSONParseError(ValueError):
    """모델 응답에서 JSON을 뽑아내지 못했을 때."""


class SchemaError(ValueError):
    """JSON은 얻었지만 요구한 구조가 아닐 때."""


# --------------------------------------------------------------------------
# 1) 지저분한 응답에서 JSON 긁어내기
# --------------------------------------------------------------------------
_FENCE = re.compile(r"```(?:json|python)?\s*(.*?)```", re.DOTALL)


def extract_json(text: str) -> str:
    """모델 응답에서 JSON 문자열만 뽑아낸다.

    소형 모델이 실제로 내놓는 형태들:
        ```json\\n{...}\\n```
        네, 알겠습니다! {...}
        {...}\\n도움이 되었길 바랍니다.

    Returns
    -------
    str : JSON 으로 보이는 부분

    Raises
    ------
    JSONParseError : 중괄호/대괄호 쌍을 찾지 못한 경우
    """
    if not text or not text.strip():
        raise JSONParseError("응답이 비어 있습니다.")

    # (1) 코드펜스가 있으면 그 안쪽을 우선 사용
    fence = _FENCE.search(text)
    if fence:
        text = fence.group(1)

    # (2) 가장 바깥쪽 { } 또는 [ ] 를 찾는다
    for opener, closer in (("{", "}"), ("[", "]")):
        start = text.find(opener)
        end = text.rfind(closer)
        if start != -1 and end > start:
            return text[start : end + 1].strip()

    raise JSONParseError(
        f"JSON 구조를 찾지 못했습니다. 응답 앞부분: {text[:100]!r}"
    )


def parse_json(text: str) -> Any:
    """extract_json + json.loads. 실패하면 JSONParseError."""
    snippet = extract_json(text)
    try:
        return json.loads(snippet)
    except json.JSONDecodeError as exc:
        # 흔한 사고: 마지막 항목 뒤의 쉼표(trailing comma)
        repaired = re.sub(r",\s*([}\]])", r"\1", snippet)
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            pass
        raise JSONParseError(
            f"JSON 파싱 실패 ({exc.msg}, {exc.lineno}행 {exc.colno}열). "
            f"문제 부분: {snippet[max(0, exc.pos - 30) : exc.pos + 30]!r}"
        ) from exc


# --------------------------------------------------------------------------
# 2) 스키마 검증
# --------------------------------------------------------------------------
def validate(data: Any, schema: dict | None) -> list[str]:
    """아주 단순한 스키마 검증기. 문제점 목록을 돌려준다 (빈 목록 = 통과).

    스키마 형식::

        {
            "required": ["test", "reason"],          # 반드시 있어야 할 키
            "types":    {"alpha": float, "n": int},  # 값의 파이썬 타입
            "allowed":  {"test": ["t-test", ...]},   # 허용되는 값 목록
            "range":    {"alpha": (0.0, 1.0)},       # 수치 범위 (닫힌 구간)
        }

    pydantic 같은 라이브러리를 쓸 수도 있지만, 여기서는 검증이 무엇을 하는
    일인지 보이도록 직접 구현한다.
    """
    if schema is None:
        return []

    problems: list[str] = []

    if not isinstance(data, dict):
        return [f"최상위가 객체(dict)가 아니라 {type(data).__name__} 입니다."]

    for key in schema.get("required", []):
        if key not in data:
            problems.append(f"필수 키 '{key}' 가 없습니다.")

    for key, expected in schema.get("types", {}).items():
        if key not in data:
            continue
        value = data[key]
        # int 는 float 자리에 들어와도 허용한다
        if expected is float and isinstance(value, int) and not isinstance(value, bool):
            continue
        if not isinstance(value, expected):
            problems.append(
                f"'{key}' 는 {expected.__name__} 여야 하는데 "
                f"{type(value).__name__} 입니다 (값: {value!r})."
            )

    for key, options in schema.get("allowed", {}).items():
        if key in data and data[key] not in options:
            problems.append(
                f"'{key}' 의 값 {data[key]!r} 는 허용 목록에 없습니다. "
                f"가능한 값: {options}"
            )

    for key, (low, high) in schema.get("range", {}).items():
        if key not in data:
            continue
        value = data[key]
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            problems.append(f"'{key}' 가 숫자가 아니어서 범위를 확인할 수 없습니다.")
        elif not (low <= value <= high):
            problems.append(f"'{key}' 의 값 {value} 가 [{low}, {high}] 범위 밖입니다.")

    return problems


# --------------------------------------------------------------------------
# 3) 프롬프트로 형식 강제하기
# --------------------------------------------------------------------------
def build_json_prompt(prompt: str, schema: dict | None = None, example: Any = None) -> str:
    """JSON 출력을 유도하는 지시문을 프롬프트에 덧붙인다.

    소형 모델에서 실제로 효과가 있었던 순서대로 넣는다.
    1) 출력 예시  2) 금지 사항 명시  3) 시작·끝 문자 지정
    """
    parts = [prompt, "", "## 출력 형식"]

    if example is not None:
        parts.append("아래와 정확히 같은 형식의 JSON으로만 답하세요.")
        parts.append(json.dumps(example, ensure_ascii=False, indent=2))
    elif schema and schema.get("required"):
        keys = ", ".join(f'"{k}"' for k in schema["required"])
        parts.append(f"다음 키를 가진 JSON 객체로만 답하세요: {keys}")

    if schema:
        for key, options in schema.get("allowed", {}).items():
            parts.append(f'- "{key}" 는 다음 중 하나여야 합니다: {options}')

    parts += [
        "",
        "규칙:",
        "- 설명, 인사, 사족을 쓰지 마세요.",
        "- 코드블록 표시(백틱)를 쓰지 마세요.",
        "- 응답은 반드시 { 로 시작해서 } 로 끝나야 합니다.",
    ]
    return "\n".join(parts)


# --------------------------------------------------------------------------
# 4) 파싱 · 검증 · 재시도를 합친 것
# --------------------------------------------------------------------------
def ask_json(
    prompt: str,
    system: str | None = None,
    *,
    schema: dict | None = None,
    example: Any = None,
    n_retry: int = 3,
    llm: Any = None,
    verbose: bool = False,
    **kw: Any,
) -> Any:
    """모델에게 JSON을 받아 파싱·검증한 결과를 돌려준다.

    실패하면 **오류 메시지를 모델에게 되돌려주고 다시 요구한다.**
    이 되먹임 구조가 6주차 ReAct 루프의 핵심 아이디어와 같다.

    Parameters
    ----------
    schema  : validate() 형식의 스키마
    example : 출력 예시 (few-shot). 소형 모델에서 가장 효과가 크다
    n_retry : 최대 시도 횟수
    llm     : LLM 인스턴스. 없으면 kmu_llm.LLM() 을 만든다
    verbose : 시도 과정을 출력

    Raises
    ------
    JSONParseError / SchemaError : n_retry 회 모두 실패한 경우
    """
    if llm is None:
        from kmu_llm import LLM

        llm = LLM()

    kw.setdefault("temperature", 0.0)  # 형식을 지켜야 하므로 낮게

    messages: list[dict[str, str]] = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": build_json_prompt(prompt, schema, example)})

    last_error: Exception | None = None

    for attempt in range(1, n_retry + 1):
        raw = llm.chat(messages, **kw)
        if verbose:
            print(f"[시도 {attempt}] 원문: {raw[:120]!r}")

        try:
            data = parse_json(raw)
        except JSONParseError as exc:
            last_error = exc
            feedback = (
                f"방금 응답을 JSON으로 읽을 수 없었습니다. 오류: {exc}\n"
                "설명 없이 JSON 객체만 다시 출력하세요."
            )
        else:
            problems = validate(data, schema)
            if not problems:
                if verbose:
                    print(f"[시도 {attempt}] 성공")
                return data
            last_error = SchemaError("; ".join(problems))
            feedback = (
                "JSON 형식은 맞지만 내용에 문제가 있습니다:\n"
                + "\n".join(f"- {p}" for p in problems)
                + "\n이 문제들을 고쳐서 JSON 객체만 다시 출력하세요."
            )

        if verbose:
            print(f"[시도 {attempt}] 실패 -> 되먹임: {last_error}")

        # 실패한 응답과 오류를 대화에 남겨서 모델이 스스로 고치게 한다
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": feedback})

    raise type(last_error)(  # type: ignore[misc]
        f"{n_retry}회 시도 모두 실패했습니다. 마지막 오류: {last_error}"
    )


In [ ]:
!pip install -q openai

# 이 수업 저장소에서 공용 모듈 내려받기

from kmu_llm import LLM, ask, check_server

PROVIDER = "kmu"          # 학과 서버가 안 되면 "gemini" 로 바꾸세요
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

---
## 1. 실패 사례 수집

프롬프트를 일부러 조금 느슨하게 주고, 온도를 올려 실패를 만들어 냅니다.
연구로 치면 **표본을 모으는 단계**입니다.

In [ ]:
import json
import pandas as pd

프롬프트 = (
    "다음 변수들을 연속형과 범주형으로 분류해서 JSON으로 답하세요: "
    "나이, 성별, 소득, 혈액형, 키, 학력수준"
)

응답들 = []
for i in range(30):
    try:
        응답들.append(llm.ask_retry(프롬프트, temperature=0.8, max_tokens=300))
    except Exception as exc:
        응답들.append(f"<호출실패: {type(exc).__name__}>")
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/30")

def 순수파싱성공(s):
    try:
        json.loads(s)
        return True
    except Exception:
        return False

성공수 = sum(순수파싱성공(r) for r in 응답들)
print(f"\njson.loads() 만으로 성공: {성공수}/30")

In [ ]:
# 실패한 것들을 눈으로 봅시다
실패들 = [r for r in 응답들 if not 순수파싱성공(r)]
print(f"실패 {len(실패들)}건 중 앞 5건:\n")
for r in 실패들[:5]:
    print(repr(r[:180]))
    print("-" * 60)

---
## 2. 유형 분류 — 규칙적인가, 아닌가

수집한 실패를 유형별로 나눕니다. **분류의 목적은 대처를 정하는 것**입니다.

| 유형 | 규칙적인가 | 대처 |
|---|---|---|
| 코드펜스로 감쌈 | 예 | 파서가 벗겨낸다 |
| 앞에 인사말 | 예 | 파서가 잘라낸다 |
| 뒤에 사족 | 예 | 파서가 잘라낸다 |
| 후행 쉼표 | 예 | 파서가 고친다 |
| 홑따옴표 | 예 | **고칠까? 말까?** ← 판단 필요 |
| 중간에 잘림 | 아니오 | `max_tokens` 를 늘리고 재시도 |
| 아예 거부/엉뚱 | 아니오 | 재시도 |

In [ ]:
import re

def 유형판정(s):
    if s.startswith("<호출실패"):
        return "호출실패"
    if "```" in s:
        return "코드펜스"
    if not re.search(r"[{\[]", s):
        return "JSON 없음"
    if s.count("{") > s.count("}"):
        return "중간에 잘림"
    if re.search(r",\s*[}\]]", s):
        return "후행 쉼표"
    if re.search(r"'\w+'\s*:", s):
        return "홑따옴표"
    if not s.lstrip().startswith(("{", "[")):
        return "앞에 사족"
    if not s.rstrip().endswith(("}", "]")):
        return "뒤에 사족"
    return "기타"

분류 = pd.Series([유형판정(r) for r in 실패들]).value_counts()
print("실패 유형 분포")
print(분류.to_string())

In [ ]:
# 유형별 실제 사례 하나씩
seen = set()
for r in 실패들:
    t = 유형판정(r)
    if t not in seen:
        seen.add(t)
        print(f"[{t}]")
        print(" ", repr(r[:160]))
        print()

---
## 3. 파서 만들기

전략은 단순합니다.

```
1) 코드펜스가 있으면 안쪽만 취한다
2) 가장 바깥쪽 { ... } 또는 [ ... ] 를 잘라낸다
3) 못 찾으면 실패로 처리한다
```

In [ ]:
FENCE = re.compile(r"```(?:json|python)?\s*(.*?)```", re.DOTALL)

class 파싱실패(ValueError):
    pass


def 내_extract_json(text):
    if not text or not text.strip():
        raise 파싱실패("응답이 비어 있습니다.")

    # (1) 코드펜스 안쪽 우선
    m = FENCE.search(text)
    if m:
        text = m.group(1)

    # (2) 가장 바깥쪽 괄호 쌍
    for opener, closer in (("{", "}"), ("[", "]")):
        start, end = text.find(opener), text.rfind(closer)
        if start != -1 and end > start:
            return text[start:end + 1].strip()

    raise 파싱실패(f"JSON 구조 없음: {text[:80]!r}")


# 테스트
테스트 = [
    '{"a": 1}',
    '```json\n{"a": 1}\n```',
    '네, 알겠습니다! {"a": 1} 도움이 되었길 바랍니다.',
    '{"a": 1}\n\n추가 설명입니다.',
    '죄송하지만 답할 수 없습니다.',
]
for t in 테스트:
    try:
        print(f"{내_extract_json(t)!r:20s}  <-  {t[:45]!r}")
    except 파싱실패 as e:
        print(f"{'실패':20s}  <-  {t[:45]!r}  ({e})")

In [ ]:
# 파서를 붙이면 성공률이 얼마나 오르나?
def 파서로파싱(s):
    try:
        json.loads(내_extract_json(s))
        return True
    except Exception:
        return False

전 = sum(순수파싱성공(r) for r in 응답들)
후 = sum(파서로파싱(r) for r in 응답들)

print(f"json.loads() 만        : {전}/30  ({전/30:.0%})")
print(f"+ extract_json()       : {후}/30  ({후/30:.0%})")
print(f"파서가 구제한 건수      : {후 - 전}건")

---
## 4. 어디까지 고쳐 줄 것인가 — 관대한 파서의 위험

후행 쉼표는 고쳐 줄까요? 홑따옴표는요?

**너무 관대하면 위험합니다.** 모델이 엉뚱한 것을 내놨는데 파서가 억지로
말이 되게 고쳐 버리면, **잘못된 결과가 조용히 통과합니다.**
통계 분석에서 이것은 치명적입니다.

In [ ]:
# 위험한 예: 홑따옴표를 무조건 큰따옴표로 바꾸면?
위험한입력 = '{"note": "it\'s fine", \'test\': \'t-test\'}'
print("원본     :", 위험한입력)
print("무식하게 :", 위험한입력.replace("'", '"'))
print()
print("-> 문자열 안의 아포스트로피까지 망가져서 의미가 달라졌습니다.")
print("   '고쳐 주는' 파서는 이렇게 조용히 데이터를 훼손할 수 있습니다.")

### 이 수업의 선택

`common/structured.py` 의 `parse_json()` 은 **후행 쉼표까지만** 고칩니다.

- 후행 쉼표: 의미를 바꿀 수 없는 순수한 문법 오류 → 고친다
- 홑따옴표, 잘린 JSON, 없는 키: 의미가 달라질 수 있음 → **실패로 처리하고 재시도**

> 원칙: **조용히 틀리는 것보다 명확하게 실패하는 편이 낫다.**

In [ ]:
# 공식 구현과 비교
from structured import extract_json, parse_json, JSONParseError

for t in 테스트 + ['{"a": 1, }']:
    try:
        print(f"{parse_json(t)!r:22s}  <-  {t[:45]!r}")
    except JSONParseError:
        print(f"{'JSONParseError':22s}  <-  {t[:45]!r}")

In [ ]:
def 성공하는가(fn, arg):
    try:
        fn(arg)
        return True
    except Exception:
        return False

공식 = sum(성공하는가(parse_json, r) for r in 응답들)
print(f"json.loads() 만          : {전}/30  ({전/30:.0%})")
print(f"내_extract_json()        : {후}/30  ({후/30:.0%})")
print(f"structured.parse_json()  : {공식}/30  ({공식/30:.0%})")
print()
print("공식 구현은 후행 쉼표까지 고쳐 주므로 조금 더 높습니다.")

---
## 5. 파싱에 성공했다고 끝이 아니다

아래 넷은 **전부 파싱에 성공합니다.** 그러나 아래 셋은 그대로 쓰면 사고가 납니다.

In [ ]:
사례들 = [
    '{"test": "t-test", "alpha": 0.05}',
    '{"test": "티검정", "alpha": 0.05}',
    '{"test": "t-test", "alpha": 5}',
    '{"test": "t-test"}',
]
for s in 사례들:
    print(f"파싱 결과: {parse_json(s)}")

print()
print("전부 성공했습니다. 하지만...")
print("  2번: '티검정' 은 scipy 에 없는 이름입니다")
print("  3번: 유의수준 5?")
print("  4번: alpha 가 없는데 다음 코드에서 참조하면 KeyError")
print()
print("-> 다음 노트북에서 '검증'을 붙입니다.")

---
## 연습문제

**문제 1.** `유형판정()` 함수를 개선하시오. 지금 "기타"로 빠지는 것들이 있는가?

**문제 2.** 온도를 0.0, 0.4, 0.8, 1.2 로 바꿔 가며 각 30회 수집하고,
**실패 유형의 분포**가 온도에 따라 달라지는지 교차표로 확인하시오.
(높은 온도에서 특히 늘어나는 유형이 있는가?)

**문제 3.** `내_extract_json()` 은 텍스트에 중괄호가 **두 번** 나오면
(예: 설명 안에 `{예시}` 가 있는 경우) 잘못 잘라냅니다.
직접 그런 입력을 만들어 확인하고, 개선안을 제시하시오.

**문제 4.** 파서로 흡수 가능한 유형과 불가능한 유형의 비율을 계산하시오.
파서를 아무리 잘 만들어도 남는 실패율은 얼마인가?

In [ ]:
# 여기에 직접 작성해 보세요

<details>
<summary><b>해설 보기</b></summary>

**문제 2.** 온도가 오르면 "앞에 사족", "JSON 없음" 이 늘어납니다.
형식 준수는 최빈 경로를 따라가는 일인데 온도가 그것을 흐트러뜨리기 때문입니다.
교차표에 `scipy.stats.chi2_contingency` 를 적용해 독립성을 검정해 볼 수 있습니다.

**문제 3.** 예: `설명 {이런 식} 이고 결과는 {"a": 1} 입니다.`
가장 바깥 `{` 부터 가장 바깥 `}` 까지 자르면 전체가 통째로 잡혀 파싱이 실패합니다.

개선 방향: 괄호 **깊이를 세면서** 짝이 맞는 첫 완결 블록을 찾는 것입니다.
```python
depth = 0
for i, ch in enumerate(text):
    if ch == "{":
        if depth == 0:
            start = i
        depth += 1
    elif ch == "}":
        depth -= 1
        if depth == 0:
            return text[start:i + 1]
```
다만 이래도 완벽하지 않습니다(문자열 안의 중괄호). **파서를 정교하게 만드는
것에는 한계가 있고, 결국 재시도가 필요하다**는 것이 요점입니다.

**문제 4.**
```python
흡수가능 = {"코드펜스", "앞에 사족", "뒤에 사족", "후행 쉼표"}
분류표 = pd.Series([유형판정(r) for r in 실패들]).value_counts()
막을수있음 = 분류표[분류표.index.isin(흡수가능)].sum()
print(f"파서로 구제 가능: {막을수있음}/{len(실패들)}")
print(f"남는 실패율: {(len(실패들) - 막을수있음) / 30:.0%}")
```
이 남는 실패율이 다음 노트북에서 재시도로 줄여야 할 몫입니다.

</details>

---
## 정리

- 실패를 **수집하고 분류**한 뒤에 대처를 정한다
- 규칙적인 실패(펜스·사족·후행쉼표)는 **파서로 흡수**
- 불규칙한 실패(잘림·거부)는 **재시도의 영역**
- **관대한 파서는 위험하다.** 조용히 틀리느니 명확히 실패하는 편이 낫다
- 파싱 성공 ≠ 사용 가능. **검증이 남았다**

**다음 노트북** → [`03_ask_json.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week03/03_ask_json.ipynb)
검증과 되먹임 재시도를 붙여 `ask_json()` 을 완성합니다.